# 09 - Hierarchical Taxonomy Clustering

Following source `08_clustering.ipynb`:
1. Read morphotope-level medians + morph_chars
2. Filter noise, drop orientation/alignment columns
3. `StandardScaler` + `nan_to_num(0)`
4. Build `KNN(k=10)` connectivity graph
5. `AgglomerativeClustering(ward)` with KNN connectivity
6. Cut dendrogram at 7 distance thresholds to produce hierarchy
7. Map cluster labels back to tessellation cells

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), "") if os.path.basename(os.getcwd()) == "notebooks" else ".")

import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph
from scipy.cluster.hierarchy import fcluster
from pathlib import Path

OUT = Path("outputs/09_hierarchy")
OUT.mkdir(parents=True, exist_ok=True)

## Load morphotope data

In [2]:
cdata = pd.read_parquet("outputs/06_morphotopes/data_morphotopes_postprocessed.parquet")
morph_chars = pd.read_parquet("outputs/08_morphotope_chars/morph_chars.parquet")

# Drop noise morphotopes
cdata = cdata[~cdata.index.str.endswith('_-1')].copy()

# Keep only median statistic (source drops percentile_25, percentile_75, mean, std)
median_cols = [c for c in cdata.columns if c[1] == 'median']
cdata_median = cdata[median_cols].copy()
cdata_median.columns = [c[0] for c in cdata_median.columns]  # flatten MultiIndex

# Drop source to_drop columns
TO_DROP = [
    'stcSAl', 'stbOri', 'stcOri', 'stbCeA',
    'ldkAre', 'ldkPer', 'lskCCo', 'lskERI',
    'lskCWA', 'ltkOri', 'ltkWNB', 'likWBB', 'likWCe',
    'licBAD', 'misBAD', 'ssbCCM', 'ssbCCD',
    'ssbCor', 'mtdDeg', 'mtbSWR',  # additional drops from source 08_clustering
    'BLDG_POP',  # population — commented out from pipeline
]
drop_existing = [c for c in TO_DROP if c in cdata_median.columns]
cdata_median = cdata_median.drop(columns=drop_existing)

# Merge morphotope chars
for col in ['limLPS', 'limAre', 'limPer']:
    if col in morph_chars.columns:
        cdata_median = cdata_median.join(morph_chars[col], how='left')

# Outlier removal (source thresholds)
if 'limAre' in cdata_median.columns:
    cdata_median = cdata_median[cdata_median['limAre'] <= 5e5]
if 'limPer' in cdata_median.columns:
    cdata_median = cdata_median[cdata_median.get('limPer', 0) <= 2e5]
    cdata_median = cdata_median.drop(columns=['limPer'], errors='ignore')
if 'sdbAre' in cdata_median.columns:
    cdata_median = cdata_median[cdata_median['sdbAre'] >= 20]

print(f"Morphotope feature matrix: {cdata_median.shape}")
print(f"Features: {list(cdata_median.columns)}")

Morphotope feature matrix: (1703, 58)
Features: ['sdbAre', 'sdbPer', 'sdbCoA', 'ssbCCo', 'ssbSqu', 'ssbERI', 'ssbElo', 'libNCo', 'ldbPWL', 'ltcBuA', 'mtbAli', 'mtbNDi', 'ltbIBD', 'stbSAl', 'mibCou', 'mibAre', 'mibLen', 'mibElo', 'mibERI', 'mibCCo', 'mibLAL', 'mibFR', 'mibSCo', 'sdcLAL', 'sdcAre', 'sscCCo', 'sscERI', 'mtcWNe', 'mdcAre', 'sicCAR', 'micBAD', 'sdsLen', 'sssLin', 'ldsMSL', 'sisBpM', 'sdsSPW', 'sdsSPO', 'sdsSWD', 'sdsAre', 'sddAre', 'mtdMDi', 'midRea', 'midAre', 'lcdMes', 'ldsCDL', 'lddNDe', 'linWID', 'linPDE', 'linP3W', 'linP4W', 'lcnClo', 'xcnSCl', 'midBAD', 'HEIGHT', 'volume', 'floor_count', 'limLPS', 'limAre']


## Ward clustering with KNN connectivity

In [3]:
# StandardScaler (source uses StandardScaler for hierarchy, not QuantileTransformer)
scaler = StandardScaler()
X_scaled = np.nan_to_num(scaler.fit_transform(cdata_median), 0)

# KNN connectivity graph (source: k=10)
N_NEIGHBORS = 10
knn = kneighbors_graph(X_scaled, n_neighbors=N_NEIGHBORS, include_self=True, mode='distance')
print(f"KNN graph: {knn.shape}, nnz={knn.nnz}")

# Ward AgglomerativeClustering
clusterer = AgglomerativeClustering(
    linkage='ward', metric='euclidean',
    connectivity=knn,
    compute_full_tree=True, compute_distances=True,
)
model = clusterer.fit(X_scaled)

# Build linkage matrix
counts = np.zeros(model.children_.shape[0])
n_samples = len(model.labels_)
for i, merge in enumerate(model.children_):
    count = 0
    for child in merge:
        count += 1 if child < n_samples else counts[child - n_samples]
    counts[i] = count
linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)

np.save(OUT / "linkage_matrix.npy", linkage_matrix)
print(f"Linkage matrix: {linkage_matrix.shape}")

KNN graph: (1703, 1703), nnz=17030
Linkage matrix: (1702, 4)


## Cut dendrogram at 7 levels

Source hierarchy shape (from the paper):
- Level 1 (coarsest): **2 clusters** — "Coherent Fabric" vs "Incoherent Fabric"  
- Level 2: **4** — Large-Scale/Small-Scale x Coherent/Incoherent
- Level 3: **8** — named types (e.g. "Coherent Dense Adjacent Fabric")
- Levels 4-7: progressively finer, up to ~110 at finest

We use `maxclust` criterion (target cluster counts) to reproduce this binary
branching pattern, scaled to our 1,716 Winnipeg morphotopes.

In [4]:
distances = linkage_matrix[:, 2]
print(f"Dendrogram distance range: {distances.min():.2f} to {distances.max():.2f}")
print()

# ============================================================
# APPROACH 1: Target cluster counts (active)
# Reproduces the source's binary branching hierarchy shape
# Source: 2 → 4 → 8 → ~15 → ~30 → ~60 → ~110
# ============================================================
TARGET_COUNTS = [2, 4, 8, 16, 32, 64, 128]

cluster_mapping = pd.DataFrame(index=cdata_median.index)

for i, n_clust in enumerate(TARGET_COUNTS):
    level = i + 1
    labels = fcluster(linkage_matrix, t=n_clust, criterion='maxclust')
    cluster_mapping[f'level_{level}'] = labels
    print(f"Level {level}: {n_clust:>3d} target -> {len(set(labels)):>3d} clusters")

# ============================================================
# APPROACH 2: Distance-based percentile cutoffs (commented out)
# Use when you want data-driven thresholds instead of fixed counts.
# Uncomment this block and comment out Approach 1 above.
# ============================================================
# PERCENTILES = [90, 93, 95, 97, 98, 99, 99.5]
# CUTOFFS = [float(np.percentile(distances, p)) for p in PERCENTILES]
#
# cluster_mapping_dist = pd.DataFrame(index=cdata_median.index)
#
# for i, cutoff in enumerate(CUTOFFS):
#     level = i + 1
#     labels = fcluster(linkage_matrix, t=cutoff, criterion='distance')
#     cluster_mapping_dist[f'level_{level}'] = labels
#     n = len(set(labels))
#     print(f"Level {level} (cutoff={cutoff:.2f}, p{PERCENTILES[i]}): {n} clusters")
#
# cluster_mapping = cluster_mapping_dist  # swap to use distance-based

# ============================================================
# APPROACH 3: Hardcoded distance cutoffs (commented out)
# Source European thresholds — only works if dendrogram max > 2500.
# For Winnipeg (max ~170) these give 1 cluster per level.
# ============================================================
# CUTOFFS_EU = [200, 300, 500, 700, 1000, 1600, 2500]
#
# cluster_mapping_eu = pd.DataFrame(index=cdata_median.index)
#
# for i, cutoff in enumerate(CUTOFFS_EU):
#     level = i + 1
#     labels = fcluster(linkage_matrix, t=cutoff, criterion='distance')
#     cluster_mapping_eu[f'level_{level}'] = labels
#     n = len(set(labels))
#     print(f"Level {level} (cutoff={cutoff}): {n} clusters")
#
# cluster_mapping = cluster_mapping_eu  # swap to use EU thresholds

cluster_mapping.to_parquet(OUT / "cluster_mapping.parquet")
print(f"\nHierarchy saved to {OUT / 'cluster_mapping.parquet'}")

Dendrogram distance range: 0.84 to 170.75

Level 1:   2 target ->   2 clusters
Level 2:   4 target ->   4 clusters
Level 3:   8 target ->   8 clusters
Level 4:  16 target ->  16 clusters
Level 5:  32 target ->  32 clusters
Level 6:  64 target ->  64 clusters
Level 7: 128 target -> 128 clusters

Hierarchy saved to outputs\09_hierarchy\cluster_mapping.parquet


## Map cluster labels back to tessellation cells

In [5]:
morph_labels = pd.read_parquet(
    "outputs/06_morphotopes/tessellation_labels_postprocessed.parquet"
)["morphotope_label"]

# Map finest level back to every tessellation cell
finest = cluster_mapping['level_1']  # level_1 = finest (cutoff=200)
morph_to_cluster = finest.to_dict()

# Assign to tessellation cells (non-noise only for now; noise handled in notebook 10)
tess_clusters = morph_labels.map(morph_to_cluster)
tess_clusters.name = 'final'

# Also map all hierarchy levels
for col in cluster_mapping.columns:
    tess_clusters_level = morph_labels.map(cluster_mapping[col].to_dict())
    tess_clusters = pd.concat([tess_clusters, tess_clusters_level.rename(col)], axis=1)

tess_clusters.to_parquet(OUT / "tessellation_clusters.parquet")
print(f"Mapped {tess_clusters['final'].notna().sum():,} cells to clusters")
print(f"  Unmapped (noise): {tess_clusters['final'].isna().sum():,}")

# Save cluster centroids for noise assignment
cdata_median.to_parquet(OUT / "cdata.parquet")
print(f"Cluster centroid data saved ({cdata_median.shape})")

Mapped 242,523 cells to clusters
  Unmapped (noise): 26,584
Cluster centroid data saved ((1703, 58))
